<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 24. Regimes and walk-forward folds

**Questions from the meeting:** (1) does performance depend on the market regime? (2) does the model still work when tested strictly in the future, in hand-picked periods that include crashes?

**Regime labels (computed only from past prices, so no look-ahead)**
- bull: Close above its 200-day moving average and less than 10% below its 252-day high
- bear: Close below the 200-day moving average and at least 15% below its 252-day high
- sideways: everything else
- warmup: the first 200 bars, where the average is not defined

**Walk-forward folds (expanding training window, hand-picked test blocks)**
- Test 2018-2019 (late-2018 selloff and recovery), 2020-2021 (COVID crash and rally), 2022-2023 (bear market), 2024-2025 (bull market).
- The model is trained only on events before the test block.
- **Purge:** training events in the 16 calendar days (about 10 trading bars) before the test start are removed, because their labels look up to 10 bars ahead and would overlap the test period.
- All 24 tickers are pooled in time, so every event in training happened before every event in the test block.

**Overfitting check:** compare F1 on the training events with F1 on the test block. A big gap means the model memorises instead of generalising.

In [ ]:
import sys, os
for m in list(sys.modules.keys()):
    if m.startswith('src'):
        del sys.modules[m]

REPO_DIR  = '/content/regime-aware-ml-trading'
PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
if not os.path.isdir(PROJ_ROOT):
    os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
else:
    os.system(f'cd {REPO_DIR} && git pull -q')
os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import pandas as pd, numpy as np, matplotlib.pyplot as plt
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

In [ ]:
from src.experiments.helpers import TICKERS, build_event_data

ALL = ['SPY','QQQ','NVDA','GOOGL','AAPL','MSFT','AMZN','META','AVGO','TSLA','COST','ADBE',
       'AMD','NFLX','INTC','CSCO','JNJ','KO','WMT','PG','CAT','F','XOM','JPM']

def regime_series(raw):
    c = raw['Close']
    ma = c.rolling(200).mean()
    dd = c / c.rolling(252, min_periods=1).max() - 1
    r = pd.Series('sideways', index=c.index)
    r[(c > ma) & (dd > -0.10)] = 'bull'
    r[(c < ma) & (dd <= -0.15)] = 'bear'
    r[ma.isna()] = 'warmup'
    return r

event_data = {}
for tk in ALL:
    try:
        event_data[tk] = build_event_data(tk)
    except Exception as e:
        print(tk, "FAILED:", e)

all_cols = sorted(set().union(*[set(d['features'].columns) for d in event_data.values()]))

parts = []
for tk, d in event_data.items():
    lab, raw = d['labeled_df'], d['raw']
    assert len(d['features']) == len(lab) == len(d['labels']), tk
    dates = pd.to_datetime(lab['event_date']).values
    reg = regime_series(raw)
    X = d['features'].reindex(columns=all_cols, fill_value=0).fillna(0).reset_index(drop=True)
    meta = pd.DataFrame({'ticker': tk, 'date': dates,
                         'y': d['labels'].values,
                         'regime': reg.reindex(pd.to_datetime(dates)).values})
    parts.append(pd.concat([meta, X], axis=1))

ev = pd.concat(parts, ignore_index=True).sort_values('date').reset_index(drop=True)
FEATS = all_cols
print("Total events:", len(ev), "| tickers:", ev.ticker.nunique(),
      "| date range:", ev.date.min().date(), "to", ev.date.max().date())

In [ ]:
tab = ev.groupby('regime').y.value_counts().unstack(fill_value=0)
tab['total'] = tab.sum(axis=1)
for c in ['long', 'short', 'no_trade']:
    tab[c + '_%'] = (100 * tab[c] / tab['total']).round(1)
print(tab.to_string())

# share of bars in each regime for SPY (for context)
spy_reg = regime_series(event_data['SPY']['raw'])
print("\nSPY share of days per regime (%):")
print((100 * spy_reg.value_counts(normalize=True)).round(1).to_string())

fig, ax = plt.subplots(figsize=(10, 3))
c = event_data['SPY']['raw']['Close']
ax.plot(c.index, c.values, color='k', lw=0.8)
colors = {'bull': 'tab:green', 'bear': 'tab:red', 'sideways': 'tab:orange'}
for rg, col in colors.items():
    m = spy_reg == rg
    ax.scatter(c.index[m], c.values[m], s=2, color=col, label=rg)
ax.set_yscale('log'); ax.legend(markerscale=5); ax.set_title('SPY coloured by regime')
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, fbeta_score

def fb05(y, p):
    return fbeta_score(y, p, labels=['long', 'short'], beta=0.5, average='macro', zero_division=0)

FOLDS = [('2018-2019', '2018-01-01', '2019-12-31'),
         ('2020-2021', '2020-01-01', '2021-12-31'),
         ('2022-2023', '2022-01-01', '2023-12-31'),
         ('2024-2025', '2024-01-01', '2025-12-31')]
PURGE = pd.Timedelta(days=16)     # about 10 trading bars + 1
SEEDS = [42, 7, 123]
MIN_N = 30

fold_rows, regime_rows, preds_store = [], [], {}
for name, start, end in FOLDS:
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    tr = ev[ev.date < s - PURGE]
    te = ev[(ev.date >= s) & (ev.date <= e)].copy()
    dummy_p = DummyClassifier(strategy='stratified', random_state=42).fit(tr[FEATS], tr.y).predict(te[FEATS])
    f1s, f1_tr, f05s = [], [], []
    for seed in SEEDS:
        rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                                    random_state=seed, n_jobs=-1).fit(tr[FEATS], tr.y)
        p = rf.predict(te[FEATS])
        f1s.append(f1_score(te.y, p, average='macro'))
        f05s.append(fb05(te.y, p))
        f1_tr.append(f1_score(tr.y, rf.predict(tr[FEATS]), average='macro'))
        if seed == SEEDS[0]:
            te['pred'] = p
        for rg, g in te.assign(p_=p).groupby('regime'):
            if len(g) >= MIN_N:
                regime_rows.append(dict(fold=name, regime=rg, seed=seed, n=len(g),
                                        f1=f1_score(g.y, g.p_, average='macro')))
    preds_store[name] = te
    fold_rows.append(dict(fold=name, n_train=len(tr), n_test=len(te),
                          train_f1=np.mean(f1_tr), test_f1=np.mean(f1s), test_f1_std=np.std(f1s),
                          test_f05_ls=np.mean(f05s),
                          dummy_f1=f1_score(te.y, dummy_p, average='macro')))

fold_res = pd.DataFrame(fold_rows)
fold_res['gap'] = fold_res.train_f1 - fold_res.test_f1
pd.options.display.float_format = '{:.3f}'.format
print(fold_res.to_string(index=False))

In [ ]:
reg_res = (pd.DataFrame(regime_rows).groupby(['fold', 'regime'])
           .agg(n=('n', 'first'), f1_mean=('f1', 'mean'), f1_std=('f1', 'std')).reset_index())
print(reg_res.to_string(index=False))
print("\n(regimes with fewer than", MIN_N, "test events in a fold are left out)")

# pooled over all folds, by regime (seed 42 predictions)
allp = pd.concat(preds_store.values())
print("\nAll folds together, by regime (seed 42):")
for rg, g in allp.groupby('regime'):
    if len(g) >= MIN_N:
        print(f"{rg:9s} n={len(g):5d}  F1 macro={f1_score(g.y, g.pred, average='macro'):.3f}  "
              f"share predicted no_trade={(g.pred == 'no_trade').mean():.2f}")

In [ ]:
def hand_fold(tk, last_train_year):
    s = pd.Timestamp(f'{last_train_year + 1}-01-01')
    own = ev[ev.ticker == tk]
    tr_own = own[own.date < s - PURGE]; te = own[own.date >= s]
    tr_pool = ev[ev.date < s - PURGE]
    out = {}
    for label, tr in [('own only', tr_own), ('pooled (time-aligned)', tr_pool)]:
        f1s = []
        for seed in SEEDS:
            rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced',
                                        random_state=seed, n_jobs=-1).fit(tr[FEATS], tr.y)
            f1s.append(f1_score(te.y, rf.predict(te[FEATS]), average='macro'))
        out[label] = (len(tr), np.mean(f1s), np.std(f1s))
    dummy = DummyClassifier(strategy='stratified', random_state=42).fit(tr_own[FEATS], tr_own.y)
    print(f"{tk}: train up to {last_train_year}, test {last_train_year+1}-2025, n_test={len(te)}, "
          f"dummy F1={f1_score(te.y, dummy.predict(te[FEATS]), average='macro'):.3f}")
    for k, (n, m, sd) in out.items():
        print(f"   {k:22s} n_train={n:5d}  F1 macro={m:.3f} +- {sd:.3f}")

hand_fold('INTC', 2022)
hand_fold('NVDA', 2024)